# 第六章：让目标仓位经过市场，落到现金账本

研究模型说：“股票 40%、国债 35%、黄金 15%，其余现金。”交易账户却不会因为这一行数字自动完成配置。它需要知道当前持有多少股、能买多少、会在哪个价格成交、未成交部分怎么办，以及扣费以后还有没有现金。若黄金最多只能成交目标数量的四分之一，策略实际承担的风险也会偏离模型。

本章围绕一个十万元教学账户，从同一组目标权重一直走到实际成交、现金、损益、交易成本分析和规模限制。资产只有股票、国债、黄金三种，交易日只有四个区间，但每笔钱都能在账本中找到位置。我们先解释目标为什么如此分配，再解释市场怎样改变它，最后判断变化是信号问题、风险问题还是执行问题。

## 0. 学习目标与边界

读完后，你应能将权重换成股数和订单，区分目标仓位与实际持仓；能用协方差解释组合风险、风险贡献和波动目标；能逐项计算价差、冲击、手续费、未成交机会成本；能核对现金与持仓构成的权益，理解风险限额为什么需要覆盖下单前、成交中和持仓后。

全部代码只依赖 NumPy，使用手写价格、固定目标和确定性教学成本。我们将三种资产视作可以按整数单位交易、没有分红和融资的现货产品。真实国债或黄金产品可能涉及合约乘数、应计利息、保证金、展期和不同结算日；这些需要明确增加到账本，不能通过给收益序列换一个名称自动获得。本章主线只做多，因此保证金与借券在概念层解释，不虚构一套未经展开的做空现金机制。

## 1. 先把账户里的对象分清楚

账户权益（net asset value，NAV）是现金加按当前价格估值的持仓。本例开盘前现金十万元，没有股票或其他资产。目标股票权重 40% 表示希望股票市值约占当前权益 40%，不是要买 40 股，也不是保证股票贡献 40% 的风险。

目标数量由目标市值除以决策参考价得到。若股票参考价 100 元，账户权益十万元，40% 对应 400 股；国债价格 50 元，35% 对应 700 单位；黄金价格 20 元，15% 对应 750 单位。如果已经持有 100 股股票，订单是再买 300 股，不是再买 400 股。这里的数量是教学交易单位，真实产品还要乘以每手股数或合约乘数，并按最小交易单位取整。

实际成交数量可能小于订单数量。目标黄金 750 单位，但流动性限制只允许成交 200，剩余 550 是未完成意图，不是已有持仓。只有成交回报才改变股票数量和现金。估值价格（mark）用于计算账户值，成交价格用于现金流；两个价格相差的部分正是执行损益的重要来源。

主线把当天开盘中间价作为到达价格（arrival price）：订单开始执行时可观察的参考价。目标权重假定已在前一收盘冻结，不利用本期开盘后的未来走势。实际系统还应保存更早的信号时刻和决策价格，这样才可将等待期间的价格变化与真正成交时的偏离分开。


In [ ]:
import numpy as np

assets = np.array(["股票", "国债", "黄金"])
initial_cash = 100_000.0
# 第 t 行是在第 t 个执行时点的参考中间价；最后一行仅用于期末估值。
mid_prices = np.array([[100.0, 50.0, 20.0],
                       [102.0, 49.0, 20.4],
                       [101.0, 50.0, 19.8],
                       [98.0, 51.0, 20.2],
                       [99.0, 50.5, 20.0]])
requested_weights = np.array([[0.40, 0.35, 0.15],
                              [0.35, 0.35, 0.20],
                              [0.25, 0.30, 0.30],
                              [0.00, 0.00, 0.00]])
initial_target = np.floor(initial_cash * requested_weights[0] / mid_prices[0]).astype(int)
for a, w, q in zip(assets, requested_weights[0], initial_target):
    print(a, f"目标权重={w:.0%}, 目标数量={q}")


## 2. 风险预算：相同资金不等于相同风险

股票和国债各投 35%，并不代表两边承担一样多的波动。假设股票年波动 20%、国债 8%、黄金 16%，那么股票同样一元仓位的单资产波动大于国债。但组合风险还取决于它们是否同时涨跌：如果股票下跌时国债上涨，国债可能降低整体风险；如果两者在通胀冲击中同时下跌，分散效果就会减弱。

设 \(w\) 为资产权重列向量，\(\Sigma\) 为在同一频率口径下的收益协方差矩阵，组合波动为

\[
\sigma_p=\sqrt{w^\top\Sigma w}.
\]

\(\Sigma_{ii}\) 是资产 \(i\) 的方差，\(\Sigma_{ij}\) 是两个资产共同变动的协方差。若用年化协方差，算出的就是年化波动。主线权重只使用 90% 资金，剩余现金在教学中没有波动；因此现金不必单独放进三资产风险矩阵。

波动目标（volatility targeting）按目标波动与预测组合波动之比缩放仓位。目标 \(\sigma^*=10\%\)，预测 \(\hat\sigma_p=12\%\)，缩放量为 \(10/12\approx0.833\)。本例禁止借钱加杠杆，所以缩放最多为 1；低风险时可以保留现金，高风险时降低所有风险仓位。这样做可能平滑风险，却无法保证实际波动恰等于 10%：历史估计有延迟，价格可能跳空，相关性也会改变。

图 6-1 使用真实指数的历史波动展示这种反馈。波动突然上升后才缩仓，会在下跌以后卖出；如果大量投资者一起执行相同规则，去杠杆还可能放大市场压力。因此它是一种风险分配约定，不是消除风险的技术。




![真实 S&P 500 的 realized 波动与波动率目标下的仓位倍数](figures/ch06_vol_targeting.png)

**图 6-1｜波动率目标自动去杠杆。** 用 60 日简单收益波动估计计算 \(0.10/\hat\sigma_t\)，截断在 \([0.2,2]\)，下一交易日采用该倍数。图内极值标注由同一序列计算。滞后估计、上下限与波动变化意味着实际组合风险不一定恰好等于目标。数据类型：公开 S&P 500 本地快照。


主线使用一份“决策前已经观测到”的合成历史收益估计协方差。收缩（shrinkage）把不稳定的样本协方差向更简单的结构靠近，本例向对角矩阵收缩 30%，减弱小样本偶然相关性的影响。对角矩阵保留各资产方差、把相关性当零；它也可能过于乐观或保守，因此收缩程度本身仍是需要样本外评估的选择。

波动估计的下限、杠杆上限和仓位变化限制同样重要。如果某个资产连续没有成交、价格被重复填充，估计波动会接近零。直接按波动倒数加仓会给最难交易的资产最大权重。应先排查陈旧价格，并限制风险缩放，而不是寄希望于除数加一个很小的数就解决经济问题。


In [ ]:
risk_rng = np.random.default_rng(6027)
annual_vol = np.array([0.20, 0.08, 0.16])
correlation = np.array([[1.0, 0.2, 0.1], [0.2, 1.0, -0.1], [0.1, -0.1, 1.0]])
teaching_cov = np.outer(annual_vol, annual_vol) * correlation
past_returns = risk_rng.multivariate_normal(np.zeros(3), teaching_cov / 252, size=80)
sample_cov = np.cov(past_returns, rowvar=False) * 252
covariance = 0.70 * sample_cov + 0.30 * np.diag(np.diag(sample_cov))

def portfolio_vol(weights, cov=covariance):
    return float(np.sqrt(weights @ cov @ weights))

def risk_target(weights, target_vol=0.10):
    # 先限制单名和总敞口，再降低波动；不重新归一化放大被截断的资产。
    weights = np.clip(np.asarray(weights, dtype=float), 0, 0.40)
    gross = np.abs(weights).sum()
    if gross > 0.95:
        weights *= 0.95 / gross
    predicted = portfolio_vol(weights)
    scale = min(1.0, target_vol / predicted) if predicted > 0 else 1.0
    return weights * scale

first_weights = risk_target(requested_weights[0])
print("约束后目标:", np.round(first_weights, 4))
print(f"预测年化波动={portfolio_vol(first_weights):.2%}")


## 3. 风险贡献：为什么逆波动不等于风险平价

边际风险贡献（marginal risk contribution，MRC）表示权重稍增一点时组合波动的变化，资产 \(i\) 的值为 \((\Sigma w)_i/\sigma_p\)。总风险贡献为

\[
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p},\qquad \sum_i RC_i=\sigma_p.
\]

\((\Sigma w)_i\) 是资产与现有组合的协方差，除以组合波动后得到边际影响；再乘当前权重，得到以波动百分点计的贡献。将 \(RC_i\) 再除以 \(\sigma_p\) 可得风险贡献份额。对冲资产的贡献可能为负：增加它降低了组合风险，这不是计算错误。

风险平价（risk parity）试图让各资产风险贡献接近。逆波动权重只看每个资产自己的波动，不看与其他资产的关系；只有在某些相关结构下才接近等风险贡献。图 6-2 将资金份额和风险贡献份额并排展示。回到主线，股票 40%、国债 35% 的资金权重很接近，风险贡献却可能差很多。是否追求贡献相等，取决于我们想分配什么风险，并非任何策略的必选目标。




![等权与逆波动权重的权重份额和风险贡献份额对比](figures/ch06_risk_contribution.png)

**图 6-2｜权重不等于风险贡献。** 两种方案的权重份额与风险贡献份额不同。风险贡献份额为 \(w_i(\Sigma w)_i/(w^T\Sigma w)\)，可因对冲关系而为负；逆波动权重也不保证各资产贡献完全相等。数据类型：项目 A 合成收益的样本协方差，用于描述而非预测风险。


In [ ]:
sigma = portfolio_vol(first_weights)
risk_contribution = first_weights * (covariance @ first_weights) / sigma
inverse_vol = 1 / np.sqrt(np.diag(covariance))
inverse_vol /= inverse_vol.sum()
for a, w, rc, iv in zip(assets, first_weights, risk_contribution, inverse_vol):
    print(a, f"权重={w:.2%}, 风险贡献={rc:.2%}, 风险份额={rc/sigma:.2%}, 逆波动权重={iv:.2%}")


均值—方差优化（mean-variance optimization）会把预测收益、风险与换手共同写进目标：

\[
\max_w\;\hat\mu^\top w-\lambda w^\top\hat\Sigma w-\eta C(w-w_{\mathrm{old}}).
\]

\(\hat\mu\) 是预测收益向量，\(\hat\Sigma\) 是预测协方差，\(w_{\mathrm{old}}\) 是交易前实际权重，\(C\) 是交易成本近似，\(\lambda\) 与 \(\eta\) 分别控制风险和交易惩罚。它还需要单名、总敞口、行业、流动性和现金约束。公式不保证估计值可靠：很小的收益预测误差可能被优化器放大成大仓位。因此我们先保留手工目标与简单风险缩放，作为未来优化器必须击败的透明基线。

因子中性（factor neutrality）进一步约束 \(B^\top w\) 接近指定值，\(B\) 表示资产对市场、利率等风险因子的暴露。净敞口为零只代表正负名义金额相抵，不保证市场贝塔为零；贝塔为零也不代表尾部、流动性和融资风险消失。主线只做多，并未声称中性，反而应明确披露其股票和利率风险。

## 4. 中间价不能成交：买卖价差与订单簿

买价（bid）是别人愿意买入的最高报价，卖价（ask）是别人愿意卖出的最低报价；二者之差为买卖价差（spread），中间价为两者平均。立即买入通常付 ask，立即卖出通常收 bid。相对于 mid，两边各损失半个价差，所以不改变市场价格也会产生交易摩擦。

订单簿（order book）按价格列出等待买卖的数量。最优卖价只有 100 股时，买 6000 股不可能都按同一价格成交；订单会逐档吃掉更高价格的卖单。图 6-3 展示静态吃单。成交均价与 mid 的差包含半价差和可见深度的消耗，却不能全部称为因果市场冲击：如果没有我们的交易，价格本来可能也会变化；真实订单簿还会补单、撤单和隐藏成交。




![市价买单逐档吃穿卖单与平均成交价相对 mid 的冲击](figures/ch06_book_walk.png)

**图 6-3｜大单为什么成交在更差的价格。** 6,000 股买单按卖价从低到高成交，最后触及 100.5，均价约为 100.258。相对中间价 100.00 的滑点约为 25.8 bp，包含价差与静态吃单深度；它不全是因果意义的市场冲击。右图用阶梯表示边际卖价。数据类型：手写报价的概念示例，未模拟补单与撤单。


市价单（market order）优先争取立即成交，承担价格不确定性；限价单（limit order）指定最差可接受价格，承担无法成交的风险。限价买单在 bid 等待，不等于免费得到便宜股票：如果价格一直上涨它可能买不到；如果知情卖家在坏消息到来时主动卖给你，买到后可能立即亏损，这就是逆向选择。

队列位置表示相同价格上你前面还有多少等待成交的数量；延迟表示从决策、下单到交易所接收和回报的时间。日线开盘价既不提供队列，也不提供撤单速度，因此主线不假装模拟限价成交。它采用直接买卖的成本模型，同时显式设置每个资产最多可成交多少，保留未成交数量。

## 5. 把教学成本拆成可解释的三项

主线的到达中间价分别为 100、50、20。单边半价差设为 5、4、8 bp，手续费为成交金额的 1 bp。冲击费率采用

\[
I_i=Y\sqrt{|Q_i|/ADV_i}.
\]

\(Q_i\) 是本次实际成交数量，\(ADV_i\) 是决策前估计的日均成交数量，二者必须同为股或同为合约；\(Y=20\) bp 是教学系数，因此 \(I_i\) 的单位也是 bp。它只借用平方根形状表达规模增加时每单位成交成本升高，不是由真实成交回归出的规律。常见的更完整经验式还会乘资产波动，参数口径必须与数据校准一致。

对买单，成交价为 mid 乘 \(1+(h_i+I_i)10^{-4}\)；卖单使用负号。\(h_i\) 是半价差 bp，\(10^{-4}\) 将 bp 转成小数。手续费按实际成交金额另扣。于是一个买入价格偏离可以拆成价差与冲击，手续费不被重复塞进价格。

参与率（participation rate）是订单数量占指定市场成交量的比例。我们把每个执行窗口的允许数量简化成 ADV 的 10%，但这不是说市场在开盘瞬间真能提供全天成交量的十分之一；它只是一个可替换的容量上限。三资产 ADV 分别为 5000、10000、2000，黄金当次最多成交 200 单位，因此第一笔 750 单位目标无法完成。


In [ ]:
half_spread_bps = np.array([5.0, 4.0, 8.0])
adv_units = np.array([5000, 10000, 2000])
fee_bps = 1.0
participation_limit = 0.10

def execution_terms(asset_index, quantity, mid, cost_scale=1.0):
    if quantity == 0:
        return mid, 0.0, 0.0, 0.0
    side = np.sign(quantity)
    half = half_spread_bps[asset_index] * cost_scale
    impact = 20.0 * np.sqrt(abs(quantity) / adv_units[asset_index]) * cost_scale
    fill_price = mid * (1 + side * (half + impact) * 1e-4)
    spread_cost = abs(quantity) * mid * half * 1e-4
    impact_cost = abs(quantity) * mid * impact * 1e-4
    fee = abs(quantity) * fill_price * fee_bps * cost_scale * 1e-4
    return fill_price, spread_cost, impact_cost, fee

for i, quantity in enumerate(initial_target):
    fill_quantity = min(quantity, int(adv_units[i] * participation_limit))
    terms = execution_terms(i, fill_quantity, mid_prices[0, i])
    print(assets[i], "目标", quantity, "最多成交", fill_quantity,
          f"成交价={terms[0]:.5f}, 价差={terms[1]:.2f}, 冲击={terms[2]:.2f}, 手续费={terms[3]:.2f}")


## 6. 执行循环：先卖、再买、最后更新现金

现在把所有概念放进一个函数。每个时点先用现有股数与当时 mid 算权益，再把风险约束后的权重换成整数目标数量；订单等于目标减现有。先执行卖单释放现金，再执行买单。每张订单受参与率限制，买入还要保证含手续费后现金足够。成交之后更新股数和现金，未成交部分留在记录里；下一时点基于最新目标重新计算，不无条件追补已经失效的旧订单。

实际系统通常还要区分订单标识、撤单、成交回报、结算现金和可用现金，并保证重复回报不会重复记账。主线只有顺序执行的一个进程，没有在途订单和结算延迟，因此省去这些状态，但绝不让未成交目标进入持仓。

记账公式是 \(cash_{new}=cash_{old}-Q P^{fill}-fee\)。\(Q\) 是带符号成交数量，买入为正、卖出为负；\(P^{fill}\) 是真实成交价；`fee` 始终非负。卖出时 \(-Q P^{fill}\) 为正，现金增加。持仓更新为 \(q_{new}=q_{old}+Q\)，随后用估值价格计算权益。只要这两个公式分别成立，每笔钱就有来处。


In [ ]:
def execute_account(start_cash=100_000.0, cost_scale=1.0,
                    participation=0.10, price_path=mid_prices):
    cash = float(start_cash)
    holdings = np.zeros(3, dtype=int)
    days, fills = [], []
    for t, requested in enumerate(requested_weights):
        mid, end_mid = price_path[t], price_path[t + 1]
        nav_before = cash + holdings @ mid
        weights = risk_target(requested)
        target = np.floor(nav_before * weights / mid).astype(int)
        orders = target - holdings
        remaining = orders.copy()
        daily_spread = daily_impact = daily_fee = 0.0
        # 卖单优先；同方向按稳定的资产顺序处理。
        order_sequence = sorted(range(3), key=lambda i: (orders[i] >= 0, i))
        for i in order_sequence:
            limit = int(adv_units[i] * participation)
            signed = int(np.sign(orders[i]) * min(abs(orders[i]), limit))
            if signed < 0:
                signed = max(signed, -int(holdings[i]))
            while signed > 0:
                fill_price, spread, impact, fee = execution_terms(i, signed, mid[i], cost_scale)
                if signed * fill_price + fee <= cash + 1e-10:
                    break
                signed -= 1
            fill_price, spread, impact, fee = execution_terms(i, signed, mid[i], cost_scale)
            cash -= signed * fill_price + fee
            holdings[i] += signed
            remaining[i] -= signed
            daily_spread += spread
            daily_impact += impact
            daily_fee += fee
            fills.append({"t": t, "asset": str(assets[i]), "order": int(orders[i]),
                          "fill": signed, "unfilled": int(remaining[i]),
                          "arrival": float(mid[i]), "price": float(fill_price),
                          "spread": spread, "impact": impact, "fee": fee})
        nav_after = cash + holdings @ mid
        price_pnl = float(holdings @ (end_mid - mid))
        nav_end = cash + holdings @ end_mid
        opportunity = float(remaining @ (end_mid - mid))
        days.append({"t": t, "target": target.copy(), "orders": orders.copy(),
                     "holdings": holdings.copy(), "remaining": remaining.copy(),
                     "cash": cash, "nav_before": nav_before, "nav_after": nav_after,
                     "nav_end": nav_end, "price_pnl": price_pnl,
                     "spread": daily_spread, "impact": daily_impact, "fee": daily_fee,
                     "net_pnl": nav_end - nav_before, "opportunity": opportunity,
                     "actual_weights": holdings * mid / nav_after})
    return days, fills

account, fills = execute_account()
for row in account:
    print("时点", row["t"], "目标数量", row["target"], "实际数量", row["holdings"],
          "剩余", row["remaining"], f"现金={row['cash']:.2f}, 期末权益={row['nav_end']:.2f}")


第一时点黄金的未完成部分使账户持有更多现金，实际黄金风险小于目标。股票和国债分别接近 40% 与 35%，黄金实际只有约 4%，比目标 15% 少了约 11 个百分点。剩余现金不是策略主动选择的防御配置，而是执行限制的结果；风险报告必须把两者区分开。最后一时点目标是零，但参与率仍然存在，清仓命令也可能无法一次完成。代码如实保留剩余持仓，并按最后一行价格估值；不能因为研究结束就假设所有资产免费瞬时清仓。报告“期末权益”与“最终已全变现现金”必须分别说明。

买入能力也与订单顺序有关。在资金不足时，先买股票可能挤占黄金预算。主线使用稳定顺序，只为让这个取舍清晰可重复；更完善的执行器可以按紧迫性或比例分配现金。无论选择哪一种，都应在实际持仓里观察效果，不能让每张订单都各自使用同一笔初始现金。

## 7. 每一分钱的损益如何对上

持仓盯市（mark-to-market）是按当前估值价格计算未平仓价值。主线在一个执行时点的 mid 交易，再以次时点 mid 估值。执行后的股数 \(q^{new}\) 获得价格变化损益

\[
PnL^{price}=\sum_i q_i^{new}(M_{i,t+1}-M_{i,t}).
\]

\(M\) 是中间估值价格，\(q^{new}\) 是实际成交后的单位数量，不是目标股数。该区间总损益满足

\[
NAV_{end}-NAV_{before}=PnL^{price}-C^{spread}-C^{impact}-C^{fee}.
\]

三个成本都是货币金额，不是收益率；左侧也是金额，因此单位一致。目标没有成交的黄金不会贡献真实价格损益。若把 `target` 放进公式，却仍使用实际现金，就会平白创造或删除财富。


In [ ]:
for row in account:
    costs = row["spread"] + row["impact"] + row["fee"]
    print(f"时点 {row['t']}: 价格损益={row['price_pnl']:.2f}, "
          f"价差={row['spread']:.2f}, 冲击={row['impact']:.2f}, 手续费={row['fee']:.2f}, "
          f"净损益={row['net_pnl']:.2f}")
print(f"总权益变化={account[-1]['nav_end'] - initial_cash:.2f}")
print(f"实际末端持仓={account[-1]['holdings']}, 末端现金={account[-1]['cash']:.2f}")


收益率则需要除以适当的期初权益。不能把股票涨 2%、国债涨 1% 和手续费 20 元直接相加。本章保留金额账本，是为了让现金守恒容易检查；比较不同规模账户时，再除以各自权益。

图 6-4 在固定价格和固定权重路径下提高成本。成本单调性是会计关系：对应交易支付更多非负成本，净值不能因此提高。但如果增加成本后买入数量减少，随后恰好避开下跌，另一条动态路径的净值可能反而更高；那是在比较不同持仓策略，不能用来否定固定路径的会计关系。Sharpe 比率也不具有同样的普遍单调性。




![同一策略在 0/10/25/50 bps 成本下的净值曲线与净 Sharpe](figures/ch06_cost_monotonicity.png)

**图 6-4｜成本单调性是不变量。** 价格与权重固定时，提高非负交易成本不能提高对应日的净收益与净值。图中净 Sharpe 也下降，但 Sharpe 的单调性不是普适会计不变量。虚线为同资产集合的等权基准。数据类型：项目 A 的确定性合成输出。


图 6-5 再将毛收益、成本拖累与净收益放到一张图上。简单收益的累加可做加法归因，复利净值则需要逐期连乘。主线使用货币金额，逐期净损益能直接相加，是因为没有外部入金出金且各区间权益首尾衔接；如果加入申购赎回，需要把外部现金流与投资损益分开。




![项目 A 10 bps 情景下毛收益、成本拖累与净收益的累计曲线](figures/ch06_pnl_decomposition.png)

**图 6-5｜PnL 分解：毛收益与成本拖累。** 图中毛收益、负的成本拖累与净收益都按逐日简单收益累加，满足加法分解；它们不是复利净值。成本为 \(\mathrm{turnover}_t c_{\mathrm{bps}}10^{-4}\)，换手按目标权重变化的绝对值和计算。数据类型：项目 A 合成数据的 10 bps 成本情景。


真实跨资产账本还要增加分红与票息、期货展期、外币汇兑、借券、融资和利息。外汇损益不是“美元资产涨跌乘一下汇率”这么简单的附注：本币资产收益与汇率变化同时决定报告币种收益。期货名义本金也不等于付出的现金，保证金只是担保，因此不能把期货头寸直接放进现货买卖现金公式。每增加一类产品，都要重新明确现金流和估值，而不是只扩展资产名称列表。

## 8. 交易成本分析：真实亏损与错失收益分开看

交易成本分析（transaction cost analysis，TCA）把执行结果与某个事前基准比较。实施差额（implementation shortfall，IS）常以决策时价格为基准，衡量理想立即完成的组合与实际执行组合的差。它既包括成交部分付出的不利价格，也包括未成交部分错过的后续变化。

对一笔买入目标 \(Q\)，到达价 \(P_0\)，实际分批成交 \(q_j\)、价格 \(P_j\)，评估终点价 \(P_T\)，简化形式为

\[
IS=\sum_j q_j(P_j-P_0)+Fee+\left(Q-\sum_jq_j\right)(P_T-P_0).
\]

\(Q,q_j\) 在这里均为买入正数量，`Fee` 是手续费金额。第一项是成交偏离，最后一项是未成交机会成本。若价格下跌，没买到反而有利，机会成本可以为负；它不是需要再从现金扣掉的一张账单。卖单可用带符号数量写同类公式，关键是保持方向一致。

主线每个时点的 `remaining` 是带符号未成交量，因此 `remaining @ (end_mid-mid)` 就是固定该次目标、到下一估值时刻的机会成本。它可以诊断为何目标与实际收益不同，但不能再次从已经正确记账的实际权益扣除，否则重复处罚了未持仓。

看一个独立手算：本来要在 100 元买 1000 股，实际只买 600 股，等待后参考 mid 到 100.6，成交 101，终点 102。成交偏离 600 元，其中等待价格变动 360 元，成交时相对 mid 偏离 240 元；未成交机会成本 800 元。未计手续费的 IS 为 1400 元。这比简单说“滑点 1%”更完整，因为后者遗漏了没买到的 400 股。


In [ ]:
for row in account:
    execution_cost = row["spread"] + row["impact"] + row["fee"]
    shortfall = execution_cost + row["opportunity"]
    print(f"时点 {row['t']}: 已成交成本={execution_cost:.2f}, "
          f"未成交机会成本={row['opportunity']:.2f}, 本次目标实施差额={shortfall:.2f}")


TCA 还可以比较成交量加权平均价（VWAP）或到达价。VWAP 把区间每笔市场成交价按成交量加权，是一种相对执行基准，不保证按它成交就满足策略目标：如果 Alpha 在前十分钟已经消失，整天 VWAP 再漂亮也可能晚了。主线使用 arrival 是为了贴近目标产生时的可观察机会，真实项目应在研究前约定基准，避免事后挑最有利的一个。

## 9. 为什么拆单：冲击与等待风险的取舍

时间加权执行（TWAP）把订单按时间均分，适合没有更好成交量信息的简单安排；成交量加权执行（VWAP 算法）按预计成交量曲线分配交易；成交参与率执行（POV）随实际市场成交量保持某个比例。它们都是执行规则，不是盈利信号。主线的数量上限更接近静态参与率约束，却没有真实日内成交流，因此不能声称实现了完整 POV。

拆慢一点通常降低瞬时吃单和冲击，但剩余头寸暴露在未来价格风险里。对买单，等待时价格上涨使交易更贵；对已有卖出意图，等待时下跌也不利。Almgren–Chriss 框架用一个目标表达这种取舍：最小化预期执行成本，加上风险厌恶参数乘执行成本方差。剩余数量越大、等待时间越长、价格波动越大，等待风险通常越高。

临时冲击（temporary impact）描述交易时流动性被消耗导致的短时价格偏离；永久冲击（permanent impact）描述交易信息或持续需求引起的较持久价格变化。这些需要逐单、逐时段数据识别，主线平方根费率没有区分两者，不能把参数解释成已估计的因果冲击。

若某信号预计半小时内减弱一半，整天慢慢买可能错过收益；若信号预期持续数周，可以更耐心地控制冲击。目标函数应该同时考虑 Alpha 衰减与交易成本，而不是为所有订单固定一个执行时长。限价单也应把成交概率和未成交机会成本放进比较，不能只比较“市价单要付半价差、限价单似乎不用付”。

## 10. 资金放大以后，策略变成什么样

保持权重相同，资金从十万元增加到一千万元，目标股数大约放大一百倍；市场 ADV 不会随账户一起放大。因此同样的参与率上限使完成比例下降，未成交风险上升。容量（capacity）是特定收益、风险和执行要求下能承载的规模，不是市场总成交量，也不是一个永久不变的常数。

下面只观察首笔交易：各规模先生成目标数量，再应用同样的市场数量上限。完成比例按成交到达名义金额除以目标调仓名义金额计算，避免拿不同价格的股数直接相加。这个表没有给出“最大可管理资金”，因为还没有约定可以忍受多少未成交、多少天完成以及收益衰减速度；它只把容量瓶颈显露出来。


In [ ]:
for scale in [100_000, 1_000_000, 10_000_000]:
    scenario, _ = execute_account(start_cash=scale)
    first = scenario[0]
    desired_notional = np.abs(first["orders"]) @ mid_prices[0]
    completed = np.abs(first["orders"] - first["remaining"]) @ mid_prices[0]
    paid = first["spread"] + first["impact"] + first["fee"]
    print(f"资金={scale:>10,.0f}, 首笔完成比例={completed/desired_notional:.2%}, "
          f"成本/权益={paid/scale:.4%}, 实际总权重={np.abs(first['actual_weights']).sum():.2%}")


成本占总权益下降不一定是好消息：大账户可能只是因为绝大部分目标根本没成交。在研究报告里，同时列完成率、实际风险、未成交机会成本和净收益，才能区分“执行便宜”与“没有执行”。如果增加参与率来完成目标，冲击模型、订单簿和成交对手又需要重新评估。

多资产也可能在同一时间争夺同一种流动性。危机里所有股票仓位一起卖出、多个策略一起去杠杆，历史平均 ADV 可能严重高估当前能成交的量。借券额度、保证金、结算现金和交易对手限额同样可能比价格冲击更早限制规模。

## 11. 风险控制要在三个时刻生效

下单前检查目标和订单：证券是否允许交易、单名与总敞口是否超限、订单数量是否超过参与率、价格是否偏离合理区间、可用现金是否足够。主线的 `risk_target` 和数量上限分别承担了一部分责任。硬限制（hard limit）意味着越界就拒单，不能只是打印提示后照常执行。

成交过程中检查订单状态：累计成交是否超过原委托、重复回报是否被重复记账、撤单之后是否又有迟到成交、连接断开后是否仍有在途订单。熔断开关（kill switch）通常首先停止新增风险、撤掉可撤订单，再按照已知实际持仓进入受控处理；它不是一个能够撤销既有亏损的按钮，也不应在不知道持仓时盲目反向下单。

持仓后检查实际风险与预测是否一致：部分成交后黄金偏低、股票偏高，组合波动和因子暴露可能偏离目标。软限制（soft limit）在接近阈值时减少新订单、降低目标或提醒复核。模型限制关注输入缺失、分布漂移和异常预测；运行限制关注时钟、行情新鲜度、权限、网络和订单幂等。不同层需要分别记录，不能用一个“风险正常”状态掩盖全部问题。

压力测试（stress testing）把状态推到历史估计之外。主线可以设股票跌 20%、国债跌 5%、黄金跌 10%；也可以将 ADV 减半、价差翻三倍，或让三个资产相关性趋近 1。前者测持仓价格风险，后者测退出成本和无法成交风险。应同时问“价格会亏多少”和“需要减仓时能否减掉”，不能只给持仓乘一个震荡向量就宣称完成流动性压力测试。


In [ ]:
first = account[0]
shock = np.array([-0.20, -0.05, -0.10])
loss = float((first["holdings"] * mid_prices[0]) @ shock)
print(f"首笔实际持仓的联合冲击损益={loss:.2f}, 占成交后权益={loss/first['nav_after']:.2%}")
print("ADV 减半时，按相同参与率的单次数量上限:",
      np.floor(adv_units * 0.5 * participation_limit).astype(int))
print("首笔持仓完全退出至少需要的执行窗口数:",
      np.ceil(first["holdings"] / np.maximum(1, np.floor(adv_units * 0.5 * participation_limit))).astype(int))


最后一个窗口数仍是乐观估计：它假设每个窗口都能用同样价格和流动性成交，没有别的投资者争抢，也不考虑新的价格风险。压力测试的价值在于暴露脆弱点和制定动作，而不是对未知危机给出精确概率。

## 12. 常见误解

**“目标已经是 40%，持仓自然也是 40%。”** 股数取整、成本、部分成交和价格变化都会造成偏离，必须按实际成交与估值重新计算。

**“逆波动就是风险平价。”** 逆波动只处理单资产风险尺度；组合风险贡献还取决于协方差。相关性改变时，旧权重可能同时暴露于一个风险来源。

**“限价单没有交易成本。”** 未成交可能错过价格变化，成交也可能受到逆向选择。成本不一定以手续费形式从现金扣走。

**“净敞口为零就没有方向风险。”** 名义金额相抵不等于市场、行业、利率或尾部暴露相抵，还可能占用融资与借券额度。

**“费用越高，任何回测指标都必然越差。”** 固定路径的净收益与净值有明确会计单调性；Sharpe 与会改变成交路径的动态策略没有同样的一般保证。

**“回测最后一行应该自动清仓。”** 清仓同样要受可交易时间、流动性和费用约束。主线保留未完成清仓，才能诚实表达账户剩余风险。

## 13. 手工练习

### 练习一：一张部分成交单怎样改变权益

账户现金 10000 元、无持仓，希望按到达 mid 100 元买 80 股。实际只买 50 股，成交价 100.10，手续费 2 元。成交后 mid 仍为 100，下一估值价为 102。计算成交后现金、成交后权益、期末权益、价格损益与净损益。

<details><summary>参考答案</summary>

现金为 \(10000-50\times100.10-2=4993\) 元。成交后权益为 \(4993+50\times100=9993\)，比交易前少 7 元，其中价格偏离 5 元、手续费 2 元。期末权益为 \(4993+50\times102=10093\)，价格损益为 100 元，净损益为 93 元，正好等于价格损益减 7 元执行成本。未成交 30 股没有进入实际权益。

</details>

### 练习二：同一订单的实施差额

延续练习一，以 80 股都按 100 元立即买到作为理想目标，以 102 元为评估终点。计算未成交机会成本和 IS。是否还应该从实际期末权益里扣除这笔机会成本？

<details><summary>参考答案</summary>

机会成本为 \(30\times(102-100)=60\) 元，IS 为 \(50\times(100.10-100)+2+60=67\) 元。理想账户期末权益为 10160 元，实际为 10093，差额正是 67。机会成本是理想与实际之间的比较项，不是实际现金支出，不能再从 10093 元中扣一次。

</details>

### 练习三：等权资金的组合波动

两资产各 50%，各自年波动 20%，相关性 0.2。计算组合年波动。若目标 10%、最大杠杆 1.2，按比例缩放后的总风险仓位是多少？

<details><summary>参考答案</summary>

方差为 \(0.5^2\times0.2^2+0.5^2\times0.2^2+2\times0.5\times0.5\times0.2\times0.2^2=0.024\)，波动约 15.49%。所需缩放为 \(0.10/0.1549\approx0.6455\)，低于 1.2 的杠杆上限，所以两资产各约 32.28%，剩余约 35.45% 现金。若缩放量大于 1.2，就截到 1.2，并承认风险目标可能无法精确达到。

</details>

## 14. 自测与思考

1. 为什么每次生成订单都必须用当前实际持仓，而不能用上次目标持仓？
2. 黄金完成率低时，为什么只看“成本/总权益”会误判执行质量？
3. 若净损益不能等于价格损益减执行成本，最先查哪些字段？
4. 在信号半小时衰减与信号持续数周两种情形下，为什么合理执行速度不同？

<details><summary>参考答案</summary>

1. 上次可能部分成交、被拒单或经历股份变化。用旧目标会把未成交当成已有持仓，导致订单方向和数量错误。
2. 没成交的部分没有支付费用，却也没有获得目标暴露。应同时看完成比例、实际风险、未成交机会成本与目标收益差。
3. 先查数量正负、手续费是否重复扣、成交价与估值价是否混用、股数是否用目标代替实际、区间起止估值是否首尾一致，以及是否遗漏外部现金流或公司行动。
4. 快速衰减的收益可能在慢慢成交之前消失，应该更重视及时完成；持续时间长的信号更有空间以较低参与率降低冲击。但两者都应比较具体的预期收益损失与执行成本，不能仅凭名称决定。

</details>

## 15. 本章小结

目标权重表达想承担的风险，订单表达改变仓位的意图，成交才改变真实股数与现金。主线从十万元账户走过风险约束、数量取整、部分成交与期末估值，让每一项收益和成本都能回到具体记录。预测组合与实际组合的差异也有了名字：已成交成本、未成交机会成本和残余风险。

风险模型负责估计与约束，执行模型负责流动性与成交，账本负责财富守恒，监控负责在假设失效时采取动作。一个候选信号只有沿这条链走完，研究报告中的净收益才具有清楚的经济含义；下一步应在真实产品和数据中逐项替换教学假设，而不是只扩大回测资金数字。
